# Run here

In [ ]:
!echo $CUDA_VISIBLE_DEVICES

In [ ]:
!gpustat

In [1]:
# languages='english'
# rewards_in_the_name = 'FR'

# from datasets import Dataset, DatasetDict, load_from_disk
from datasets import load_dataset
import pandas as pd
import evaluate
import json
import re
import numpy as np
from PIL import Image
# import pytesseract
from ddgs import DDGS
from ddgs.exceptions import DDGSException, TimeoutException  # depending on your ddgs version
from qwen_vl_utils import process_vision_info

# dataset = load_dataset('HF?DATASET/DATASET?PATH',split='train')
dataset1 = load_dataset('HF?DATASET/DATASET?PATH',split='train')

dataset1 = dataset1.add_column("language_encoded", dataset1["Language"])
dataset1 = dataset1.class_encode_column("language_encoded")
dataset1 = dataset1.filter(lambda row: row['Language']=='english', num_proc=4)

# dataset_split = dataset.train_test_split(test_size=0.3, seed=42, stratify_by_column="language_encoded")
# split_dataset = dataset.train_test_split(test_size=0.3, seed=42)
split_dataset1 = dataset1.train_test_split(test_size=0.3, seed=42, stratify_by_column="language_encoded")


# train_dataset = split_dataset["train"]
# test_dataset = split_dataset["test"]

train_dataset1 = split_dataset1["train"]
test_dataset1 = split_dataset1["test"]

format_rewards = []
accuracy_rewards=[]
inconsis_penaltys=[]
tool_debate_rewards=[]

tool_count = []

In [2]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a MCQ question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the correct option. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> correct option here </answer>"
)

In [4]:
def format_helper_userinput(example):
    # def format_helper(example):
    # Language-specific option label mapping
    option_label_map = {
        "english": {
            "a":"a",
            "b":"b",
            "c":"c",
            "d":"d"
        },
        "hindi": {
            "a": "क",
            "b": "ख",
            "c": "ग",
            "d": "घ"
        },
        "bengali": {
            "a": "ক",
            "b": "খ",
            "c": "গ",
            "d": "ঘ"
        },
        "marathi": {
            "a": "क",
            "b": "ख",
            "c": "ग",
            "d": "घ"
        },
        "gujarati": {
            "a": "ક",
            "b": "ખ",
            "c": "ગ",
            "d": "ઘ"
        },
        "tamil": {
            "a": "௧",
            "b": "௨",
            "c": "௩",
            "d": "௪"
        }
    }

    # Normalize language and answer key
    language = example.get("Language", "").strip().lower()
    # answer_key = example.get("Final Answer", "").strip().lower()  # a/b/c/d
    native_answer = example.get("Final Answer")
    lang_map = option_label_map.get(language, {})
    reverse_map = {v: k for k, v in lang_map.items()}  # native → a/b/c/d

    # Resolve option key
    option_key_letter = reverse_map.get(native_answer)

    option_text = (
        example.get(f"Option {option_key_letter}")
        if option_key_letter else None
    )

    user_input = (
        f"Question: {example['Question']}\n"
        f"Options: "
        f"{lang_map.get('a','a')}. {example['Option a']}, "
        f"{lang_map.get('b','b')}. {example['Option b']}, "
        f"{lang_map.get('c','c')}. {example['Option c']}, "
        f"{lang_map.get('d','d')}. {example['Option d']}."
    )

    return {
        "user_input": user_input,
        "native_answer_label": native_answer,
        "option_key": option_key_letter,
        "answer_text": option_text
    }
    
def format_data1(example):
    templist = []
    if len(example["images"]) !=0:
        templist = example["images"].copy()
    result_format_helper = format_helper_userinput(example)
    user_input = result_format_helper['user_input']
    if example['Final Answer']==None:
        sol = f"<think>{example['Reasoning']}</think><answer>None</answer>"
    else:
        sol = (
        f"<think>{example['Reasoning']}</think>"
        f"<answer>{result_format_helper['native_answer_label']}. {result_format_helper['answer_text']}</answer>"
        )

    returned_dict = [
          {
              "role": "system",
              "content": [
                  {
                      "type": "text",
                      "text": SYSTEM_PROMPT
                  }
              ],
          },
          {
              "role": "user",
              "content": [(
                      [{"type": "image"} for _ in example["images"]] +
                            [{"type": "text", "text": user_input}]
                            )
                #   {
                #       "type": "text",
                #       "text": user_input,
                #   }
              ],
          },
      ]
      
    return {
      "images": templist,
      "prompt": returned_dict,
        "solution":sol,
      }

In [5]:
# train_dataset = [format_data(example) for example in train_dataset]
train_dataset=[format_data1(example) for example in train_dataset1]

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration,AutoProcessor,AutoTokenizer 
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
processor = AutoProcessor.from_pretrained(model_id, use_fast=True, padding_side="left")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path=model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    # use_cache= False
)

# model.save_pretrained("Qwen2.5-VL-7B-Instruct/")
# tokenizer.save_pretrained("saved_model/")

# model.config.use_cache = False

In [ ]:
debater1 = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path=model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    # use_cache= False
)

debater2 = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path=model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    # use_cache= False
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj", "k_proj"],
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

In [12]:
import re


def format_reward(completions, solution, **kwargs):
    """Reward function that checks if the completion has a specific format."""
    # pattern = r"^<think>.*?</think>\s*<answer>.*?</answer><think>.*?</think>\s*<answer>.*?</answer>$"
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"
    
    matches = [re.match(pattern, content[0]['content'], re.DOTALL | re.MULTILINE) for content in completions]
    rewards = [1.0 if match else 0.0 for match in matches]
    format_rewards.append(max(rewards))
    return rewards

In [13]:
def accuracy_reward(completions, solution, **kwargs):
    rewards = []
    # for content in completions:
    #     answers = re.findall(r"<answer>(.*?)</answer>", content[0]['content'], flags=re.DOTALL)
    #     # options = []
    #     # print(answers)
    #     # print(solution)
    #     if len(answers)==2:
    #         if answers[0] == answers[1]:
    #             rewards.append(1)
    #         else:
    #             rewards.append(0)
    #     # for ans in answers:
    #     #     m = re.match(r"\s*([a-zA-Z])\.", ans)  # capture 'a.' or 'b.' etc.
    #     #     if m:
    #     #         options.append(m.group(1)) 
    #     # if options[0]==options[1]:
    #     #     rewards.append(1)
    #     else:
    #         rewards.append(0)
    pattern = r"<think>(.*?)</think>\s*<answer>(.*?)</answer>"
    for completion, sol in zip(completions, solution):
        # print("SOLUTION:",sol)
        # print("COMPLETION:", completion)
        sol_match = re.search(pattern, sol, flags=re.DOTALL | re.MULTILINE)
        gt_answer='None'
        pred_answer='None'
        if sol_match:
            gt_answer = sol_match.group(2).strip().lower()
        match=re.search(pattern, completion[0]['content'], flags=re.DOTALL | re.MULTILINE)
        if match:
            pred_answer=match.group(2).strip().lower()
        if gt_answer==pred_answer:
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    # print(rewards)
    accuracy_rewards.append(max(rewards))
    # accuracy_rewards.append(sum(rewards)/len(rewards))
    
    return rewards

In [14]:
def get_last_number(s):
    nums = re.findall(r"-?\d+(?:\.\d+)?", s)   # matches integers & floats
    return nums[-1] if nums else None

    
def inconsistency_penalty(completions, solution, **kwargs):
    rewards=[]
    for completion in completions:
    # Extract <think> ... </think> content
        think_match = re.search(r"<think>(.*?)</think>", completion[0]['content'], re.DOTALL)
        think_text = think_match.group(1).strip() if think_match else ""
    
        # Extract <answer> ... </answer> content
        answer_match = re.search(r"<answer>(.*?)</answer>", completion[0]['content'], re.DOTALL)
        answer_text = answer_match.group(1).strip() if answer_match else ""
    
        # Function to get last occurring number in a string
    
        last_think = get_last_number(think_text)
        last_answer = get_last_number(answer_text)
        
        if last_think is None or last_answer is None:
            rewards.append(0)
        elif last_think != last_answer:
            rewards.append(-1)
        else:
            rewards.append(0)
    
    inconsis_penaltys.append(max(rewards))
    return rewards

In [15]:
    # promptss = kwargs.get("prompts", None)
    # images = kwargs.get("images", None)
    # print(model)
    # print(promptss, images, training_model)

In [16]:
def run_zero_shot(messages,image_inputs,input_model=debater1):
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to("cuda")
    
    # Inference: Generation of the output
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
        generated_ids = input_model.generate(**inputs, max_new_tokens=256,do_sample=True,temperature=0.7)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

In [17]:
# def ocr_image(image_path):
#     img = Image.open(image_path)
#     text = pytesseract.image_to_string(img)
#     return text.strip()

def describe_image(image_inputs=None):
    # image = Image.open(image_path)
    if image_inputs ==None:
        return ''
    messages = [
            {"role": "system", "content": "Describe the image."},
            {
                "role": "user",
                "content": (
                      [{"type": "image", "image": x_factor} for x_factor in image_inputs] +
                            [{"type": "text", "text": "What is in this image(s)?"}]
                  ),
            },
        ]
    # messages = [
    #     {"role": "system", "content": "Describe the image."},
    #     {"role": "user", "content": "What is in this image?"}
    # ]
    return run_zero_shot(messages, image_inputs)


# from ddgs import DDGS
# from ddgs.exceptions import DDGSException, TimeoutException  # depending on your ddgs version

def web_search(query: str, max_results: int = 3) -> str:
    if query is None:
        return ""
    query = str(query).strip()
    if not query:
        return ""

    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
    except (DDGSException, TimeoutException, Exception) as e:
        # IMPORTANT: never crash training because retrieval failed
        return ""

    if not results:
        return ""

    bodies = []
    for r in results:
        body = r.get("body") or ""
        body = body.strip()
        if body:
            bodies.append(body)

    return "\n".join(bodies)

# def web_search(query):
#     with DDGS() as ddgs:
#         results = list(ddgs.text(query, max_results=3))
#     return "\n".join([r["body"] for r in results])

def calculate(expression):
    return_val = 'Can only evaluate expression not equations'
    try:
        expression = expression.replace(",", "")
        return str(eval(expression))
    except Exception:
        return return_val

- Use tools ONLY when they are necessary.

In [18]:
def tool_selector_model(user_question, agent_response, images_input, input_model):
    prompt_for_tool_selector = '''You are a tool-selection assistant expert in financial domain.
You have access to the following tools:
- describe_image(images: list of images to describe) - Generate a description of an image using a vision-language model;
- web_search(query: Search query string)-Search the web and return summarized text results;
- calculate(expression: Mathematical expression to evaluate (e.g., '2 + 3 * 4'))-Evaluate a mathematical expression and return the result;

Your task:
1. Read the user question and response carefully.
2. Decide whether one or more tools are required for making argument in favour or against the generated response.
3. If tools are required, select the appropriate tools.
4. If no tool is required, return {}.

Rules:
- You may call MULTIPLE tools if the task requires it.
- Do NOT hallucinate tools or parameters.
- Do NOT explain your reasoning.
- If calling tools, respond ONLY with JSON and nothing else.
- Preserve the logical order of tool execution.

Output formats:
{
  "tool_calls": [
    {
      "name": "<tool_name_1>",
      "arguments": {
        "<param1>": "<value1>"
      }
    },
    {
      "name": "<tool_name_2>",
      "arguments": {
        "<param1>": "<value1>"
      }
    }
  ]
}

No tool required:
Respond with {}'''
    tool_selector_user_input = f"{user_question}, Response: {agent_response}"
    if images_input ==None:
        tool_messages = [
                {"role": "system", "content": prompt_for_tool_selector},
                {
                    "role": "user",
                    "content": [{"type": "text", "text": tool_selector_user_input}]
                },
            ]
    else:
        tool_messages = [
                {"role": "system", "content": prompt_for_tool_selector},
                {
                    "role": "user",
                    "content": (
                          [{"type": "image", "image": x_factor} for x_factor in images_input] +
                                [{"type": "text", "text": tool_selector_user_input}]
                      ),
                },
            ]
    
    return run_zero_shot(tool_messages,images_input,input_model)

In [19]:
def argument_provider(user_question, result, agent_response, images_input, input_model):
    prompt_for_argument = f'''You are an financial expert. You need to provide the argument in favour of or against the question-response pair. 
The output of tools are provided with the prompt. 
Use them as the fact unless you find it wrong. Mention it when you find the fact wrong. Give the argument with the help of tool's output.
Outputs : 
{result}

Please provide the output in this format:
Answer: <correct option, according to argument>
Arguement: <argument in favour of or against the response generated for the question>
'''
    tool_selector_user_input = f"{user_question}, Response: {agent_response}"

    if images_input ==None:
        tool_messages = [
                {"role": "system", "content": prompt_for_argument},
                {
                    "role": "user",
                    "content": [{"type": "text", "text": tool_selector_user_input}]
                },
            ]
    else:
        tool_messages = [
                {"role": "system", "content": prompt_for_argument},
                {
                    "role": "user",
                    "content": (
                          [{"type": "image", "image": x_factor} for x_factor in images_input] +
                                [{"type": "text", "text": tool_selector_user_input}]
                      ),
                },
            ]

    # tool_messages = [
    #         {"role": "system", "content": prompt_for_argument},
    #         {
    #             "role": "user",
    #             "content": (
    #                   [{"type": "image", "image": x_factor} for x_factor in images_input] +
    #                         [{"type": "text", "text": tool_selector_user_input}]
    #               ),
    #         },
    #     ]
    return run_zero_shot(tool_messages,images_input,input_model)

In [20]:
# def tool_implement(calls,images_input):
#         # print(calls)
#     results = []

#     for call in calls.get("tool_calls", []):
#         tool = call["name"]
#         args = call.get("arguments", {})

#         # if tool == "retrieve_docs":
#         #     result = retrieve_docs(**args)

#         # elif tool == "ocr_image":
#         #     result = ocr_image(**args)

#         if tool == "describe_image":
#             result = describe_image(images_input)

#         elif tool == "web_search":
#             result = web_search(**args)

#         elif tool == "calculate":
#             result = calculate(**args)

#         else:
#             raise ValueError(f"Unknown tool: {tool}")

#         results.append({
#             "tool": tool,
#             "arguments": args,
#             "result": result
#         })

#     return results

def tool_implement(calls, images_input):
    results = []

    # calls can be either:
    # 1) dict: {"tool_calls": [ ... ]}
    # 2) list: [ ... ]
    if isinstance(calls, dict):
        tool_calls = calls.get("tool_calls", [])
    elif isinstance(calls, list):
        tool_calls = calls
    else:
        tool_calls = []  # mask unexpected type

    for call in tool_calls:
        # support different key styles safely
        tool = call.get("name") or call.get("tool")
        args = call.get("arguments", {}) or {}
        tool_count.append(tool)
        # if arguments sometimes arrive as a JSON string
        if isinstance(args, str):
            try:
                import json
                args = json.loads(args)
            except Exception:
                args = {}

        try:
            if tool == "describe_image":
                result = describe_image(images_input)

            elif tool == "web_search":
                result = web_search(**args)

            elif tool == "calculate":
                result = calculate(**args)

            else:
                # MASK unknown tool: don't crash training
                result = None

        except Exception:
            # MASK execution errors: don't crash training
            result = None

        results.append({
            "tool": tool,
            "arguments": args,
            "result": result
        })

    return results


In [21]:
def debator_generation(user_input, response, images_input, model):
    toools = tool_selector_model(user_input, response, images_input, model)

    try:
        tools = json.loads(toools)
        tool_output = tool_implement(tools,images_input)
        # call = json.loads(response)
        # result = execute_tool(call)
    
        # messages.extend(result)
        # message = messages[1:]
        final_response = argument_provider(user_input, tool_output, response, images_input,model) 
        # final_message=[    
        #     {"role": "system", "content": tool_user_system_prompt},
        # ]
    
        # final_message.extend(message)
        # print(final_message)
        # final_response = run_qwen(final_message)
        # print(final_response)
        return final_response
    except json.JSONDecodeError:
        return ''
    

In [22]:
def listener_generation(user_question, debate_args1, debate_args2, images_inputs, input_model):
    prompt_for_argument = f'''You are an financial expert. You need to provide the answer of the question on the basis of arguments.
Following are the argument provided:
Argument1 : 
{debate_args1}
Argument2:
{debate_args2}

Please provide the output in this format:
Answer: <correct option, according to arguments>
Confidence: <on scale of 0 to 1, how confident are you on the output.>

For example:
Answer: a. Only VAT is correct
Confidence: 0.96

Listen the arguements correctly, then provide the output as per the format corresponding to the question.
'''
    
    tool_selector_user_input = f"{user_question}"

    if images_inputs ==None:
        tool_messages = [
                {"role": "system", "content": prompt_for_argument},
                {
                    "role": "user",
                    "content": [{"type": "text", "text": tool_selector_user_input}]
                },
            ]
    else:
        tool_messages = [
                {"role": "system", "content": prompt_for_argument},
                {
                    "role": "user",
                    "content": (
                          [{"type": "image", "image": x_factor} for x_factor in images_inputs] +
                                [{"type": "text", "text": tool_selector_user_input}]
                      ),
                },
            ]

    # tool_messages = [
    #         {"role": "system", "content": prompt_for_argument},
    #         {
    #             "role": "user",
    #             "content": (
    #                   [{"type": "image", "image": x_factor} for x_factor in images_input] +
    #                         [{"type": "text", "text": tool_selector_user_input}]
    #               ),
    #         },
    #     ]
    return run_zero_shot(tool_messages,images_inputs,input_model)

In [23]:
def extract_user_text(messages):
    texts = []
    for message in messages:
        if message["role"] == "user":
            for block in message.get("content", []):
                if block["type"] == "text":
                    texts.append(block["text"])
    return texts


In [24]:
def parse_ans_conf(text_input):
    answer_match = re.search(r"Answer:\s*(.+)", text_input)
    confidence_match = re.search(r"Confidence:\s*([0-9]*\.?[0-9]+)", text_input)

    answer = answer_match.group(1).strip().lower() if answer_match else None
    confidence = float(confidence_match.group(1)) if confidence_match else 0

    return answer, confidence


In [25]:
def tool_debate_reward(completions, solution, **kwargs):
    # promptss = kwargs.get("prompts", None)
    # images = kwargs.get("images", None)
    # print(model)
    # print(promptss, images, training_model)
    rewards=[]
    pattern = r"<think>(.*?)</think>\s*<answer>(.*?)</answer>"
    user_input_question = kwargs.get("prompts", None)
    images_all_all = kwargs.get("images", None)
    images_all = images_all_all[0].copy()
    # print(user_input_question)
    question_part_only  = extract_user_text(user_input_question[0])
    for completion, sol in zip(completions, solution):
        debate1_argument = debator_generation(question_part_only, completion[0]['content'], images_all,debater1)
        debate2_argument = debator_generation(question_part_only, completion[0]['content'], images_all,debater2)

        final_response = listener_generation(question_part_only, debate1_argument, debate2_argument, images_all, debater1)

        final_answer_response, confidence_score = parse_ans_conf(final_response)

        sol_match = re.search(pattern, sol, flags=re.DOTALL | re.MULTILINE)
        gt_answer='None'
        # pred_answer='None'
        if sol_match:
            gt_answer = sol_match.group(2).strip().lower()
        

        if final_answer_response==gt_answer:
            rewards.append(confidence_score)
        else:
            rewards.append(0)
    tool_debate_rewards.append(max(rewards))

    return rewards
    

In [26]:
def generate_subqueries(user_input,think_text,retrieved_data=[],images_inputs=None):
    prompt_for_subquery = f'''You are a Subquery Generator for an iterative Retrieval-Augmented Generation (RAG) system for MCQ answering.
Your task: generate EXACTLY ONE natural-language SEARCH QUERY that will retrieve authoritative evidence needed to verify the reasoning_trace and select the correct option.

You will be given:
- Question (MCQ with options)
- reasoning_trace (current hypothesis)
- retrieved_content (may be empty)

STRICT RULES:
1) Output ONLY ONE subquery as plain text.
2) DO NOT output explanations, analysis, answers, option letters, SQL, code, JSON, or tool calls.
3) The subquery must be suitable for a web or document search engine.
4) Include key legal identifiers (e.g., Article numbers, Constitution name, list names) when relevant.
5) Focus on verifying or falsifying the reasoning_trace.

OUTPUT FORMAT:
- A single line of text.
- No numbering, no bullets, no extra text.
'''
    
    subquery_user_input = f"{user_input},\n reasoning_trace: {think_text},\n retrieved_content:{retrieved_data} "

    if images_inputs ==None:
        tool_messages = [
                {"role": "system", "content": prompt_for_subquery},
                {
                    "role": "user",
                    "content": [{"type": "text", "text": subquery_user_input}]
                },
            ]
    else:
        tool_messages = [
                {"role": "system", "content": prompt_for_subquery},
                {
                    "role": "user",
                    "content": (
                          [{"type": "image", "image": x_factor} for x_factor in images_inputs] +
                                [{"type": "text", "text": subquery_user_input}]
                      ),
                },
            ]

    return run_zero_shot(tool_messages,images_inputs)
    # return run_zero_shot(tool_messages,images_inputs,input_model)

In [27]:
def retrieve_data(user_input_question,think_text,retrieved_data=[],image_input=None):
    subquery_to_ask = generate_subqueries(user_input_question,think_text,retrieved_data,image_input)
#+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    retrieved_text_web = web_search(subquery_to_ask) #Call RAG function here....................
#+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
    retrieved_data.append(retrieved_text_web)
    return retrieved_data

In [28]:
def decide_rag_prompt(user_input_question,think_text,retrieved_data=[],images_inputs=None):
    #Use RAG Here...
    # call function to generate subqueries
    prompt_for_subquery = '''You are a polyglot Financial Domain Expert. 
You need to give the answer of the MCQ question with the help provided additional information.
You will be given:
- Question (MCQ with Options)
- reasoning_trace (current hypothesis)
- retrieved_content (may be empty)

Please provide the output in this format:
Answer: <correct option, according to arguments>

For example:
Answer: a. Only VAT is correct
'''
    
    subquery_user_input = f"{user_input_question},\n reasoning_trace: {think_text},\n retrieved_content:{retrieved_data} "

    if images_inputs ==None:
        tool_messages = [
                {"role": "system", "content": prompt_for_subquery},
                {
                    "role": "user",
                    "content": [{"type": "text", "text": subquery_user_input}]
                },
            ]
    else:
        tool_messages = [
                {"role": "system", "content": prompt_for_subquery},
                {
                    "role": "user",
                    "content": (
                          [{"type": "image", "image": x_factor} for x_factor in images_inputs] +
                                [{"type": "text", "text": subquery_user_input}]
                      ),
                },
            ]


    answer_this_turn=run_zero_shot(tool_messages,images_inputs)
    answer_match = re.search(r"Answer:\s*(.+)", answer_this_turn)
    answer = answer_match.group(1).strip().lower() if answer_match else None

    return answer
    

    #call function to do webquery

In [29]:
def turns_to_reward(turn_list, maxTurn):
    return_rewards=[]
    for i in turn_list:
        temp_reward = 1-(i/maxTurn)
        return_rewards.append(temp_reward)

    return return_rewards

In [30]:
def multi_turn_rag(completions, solution, **kwargs):
    T=3
    #extracting <think> ... </think> 
    rewards=[]
    no_of_turns_per_comp=[]
    pattern = r"<think>(.*?)</think>\s*<answer>(.*?)</answer>"
    user_input_question = kwargs.get("prompts", None)
    images_all_all = kwargs.get("images", None)
    images_all = images_all_all[0].copy()
    question_part_only  = extract_user_text(user_input_question[0])
    
    for completion, sol in zip(completions, solution):
        think_match = re.search(r"<think>(.*?)</think>", completion[0]['content'], re.DOTALL)
        think_text = think_match.group(1).strip() if think_match else ""
        retrieved_data = []
        # print(user_input_question)
        gt_answer='None'
            # pred_answer='None'
        sol_match = re.search(pattern, sol, flags=re.DOTALL | re.MULTILINE)    
        if sol_match:
            gt_answer = sol_match.group(2).strip().lower()
    # for completion in completions:
        # count_of_turn = 0
        for rag_turn in range(T+1):
            turn_wise_ans = decide_rag_prompt(user_input_question,think_text,retrieved_data,images_all)
            if turn_wise_ans == gt_answer:
                # no_of_turns_per_comp.append(count_of_turn)
                no_of_turns_per_comp.append(rag_turn)
                break
            if rag_turn!=T:
                retrieved_data = retrieve_data(user_input_question,think_text,retrieved_data,images_all)
            else:
                no_of_turns_per_comp.append(rag_turn)
            # count_of_turn = count_of_turn+1
        # if count_of_turn ==T+1:
        #     no_of_turns_per_comp.append(T)

    rewards = turns_to_reward(no_of_turns_per_comp,T)
    return rewards
            

In [31]:
def adaptive_reward(completions, solution, **kwargs):
    # reward1 = format_reward(completions, solution, **kwargs)
    # reward2 = accuracy_reward(completions, solution, **kwargs)
    # reward3 = inconsistency_penalty(completions, solution, **kwargs)
    
    # aux_reward = []
    # for i in range(len(completions)):
    #     aux_r = 0.2*reward1[i] + 0.6*reward2[i] + 0.2*reward3[i] + 0.2
    #     aux_reward.append(aux_r)

    # if max(aux_reward)>=0.5:
    #     return aux_reward
        
    reward4 = tool_debate_reward(completions, solution, **kwargs)
    reward5 = multi_turn_rag(completions,solution, **kwargs)
    final_reward=[]
    for i in range(len(completions)):
        # fin_rew = 0.5* aux_reward[i]+0.25* reward4[i]+0.25*reward5[i]
        fin_rew = 0.5* reward4[i]+0.5*reward5[i]
        final_reward.append(fin_rew)
    # print(f"r1= {reward1},r2= {reward2},r3= {reward3},r4= {reward4},r5= {reward5}\n")
    # print(final_reward)
    return final_reward
        

### Need to add 'shuffle=True'

In [32]:
from trl import GRPOConfig

# Configure training arguments using GRPOConfig
training_args = GRPOConfig(
    output_dir="R4_text-Qwen2.5-VL-3B-mcq_eng",
    learning_rate=1e-5,
    remove_unused_columns=False,  # to access the solution column in accuracy_reward
    num_train_epochs=3,
    bf16=True,
    # Parameters that control the data preprocessing
    per_device_train_batch_size=16,
    # per_device_train_batch_size=2,
    
    max_completion_length=256,  # default: 256
    num_generations=2,  # default: 8
    # num_generations=8,  # default: 8
    # generation_batch_size=8,
    # max_prompt_length=2048,
    max_prompt_length=10000,
    # use_cache=False,                     # trainer-level setting ✔
    # do_shuffle=True,
    # Parameters related to reporting and saving
    report_to=None,
    logging_steps=10,
    push_to_hub=False,
    save_strategy="epoch",
    # use_vllm=True,
    # vllm_mode="colocate",
    # vllm_gpu_memory_utilization=0.50,
    # save_steps=100,
)

In [ ]:
from trl import GRPOTrainer
import json

trainer = GRPOTrainer(
    model=model,
    processing_class=processor,
    reward_funcs=[adaptive_reward],
    # reward_funcs=[format_reward],
    args=training_args,
    train_dataset=train_dataset,
    # use_vllm=True,
    # rollout_func=my_rollout,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(training_args.output_dir)
# trainer.push_to_hub(dataset_name=dataset_id)

In [ ]:
import pandas as pd
df = pd.DataFrame({
    "tool_count":tool_count,
    # "accuracy_rewards":accuracy_rewards,
    # "inconsistency_penalty":inconsis_penaltys,
    # "tool_debate_reward": tool_debate_rewards
    # "question": questions,
    # "ground_truth_answer": references_answer,
    # "ground_truth_reasoning": references_reasoning,
    # "prediction_answer": newpredictions,
    # "prediction_reasoning": newpredictions_reasoning,
    # "bert_score_f1_answer": F1.tolist(),
    # "bert_score_f1_reasoning": F12.tolist()
    
})
df.to_csv("tool_count_eng.csv", index=False)

In [ ]:
print('done')

# END OF CODE